# Qwenで日英音声ペアを作る

JESCとKFTTから音声ペアを生成し、Whisperで品質を確認します。生成用のGPUを選び、必要なセルを順に実行してください。

| 状況 | 実行順 |
| --- | --- |
| 初回 | **1〜5 → 6** で準備と5ペアの確認 |
| 本番開始・途中再開 | **1〜5 → 7 → 8**。Driveの進捗から続行 |
| 途中集計 | **9** |
| 最終集計 | **10**。全shardの完了を確認した後に実行 |

詳しい運用手順はガイドにあります。 [Qwen運用ガイド](https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus/blob/main/docs/qwen-runbook.md)


## 1〜4. Drive接続・コード取得・依存関係・GPU確認

下の4つのコードセルを上から実行します。音声と進捗は `MyDrive/ja-en-direct-s2st-corpus-data/` に保存します。


In [ ]:
# 1. Google Driveをマウントし、永続保存先を設定
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

REPO_URL = 'https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus.git'
REPO_DIR = Path('/content/ja-en-direct-s2st-corpus')
DATA_ROOT = Path('/content/drive/MyDrive/ja-en-direct-s2st-corpus-data')
HF_HOME = Path('/content/huggingface')
MODEL_ROOT = Path('/content/s2st-models')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['S2ST_DATA_ROOT'] = str(DATA_ROOT)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['S2ST_MODEL_ROOT'] = str(MODEL_ROOT)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Data:', DATA_ROOT)
print('Local model cache:', HF_HOME)

In [ ]:
# 2. 最新コードを取得
import subprocess
import sys
PYTHON = sys.executable

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
CONFIG = REPO_DIR / 'configs' / 'colab-pro.yaml'
print('Revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# 3. Colab標準のPython 3.13へ依存関係を導入（ランタイム作成ごとに1回）
subprocess.run([PYTHON, '--version'], check=True)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(
    ['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1', 'sox'],
    check=True,
)
subprocess.run(
    [
        PYTHON, '-m', 'pip', 'install',
        'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ],
    check=True,
)
subprocess.run(
    [PYTHON, '-m', 'pip', 'install', '-r', 'requirements/colab.txt'],
    check=True,
)
subprocess.run(
    [PYTHON, '-m', 'pip', 'install', '--no-deps', '-e', '.'],
    check=True,
)
# Transformers imports optional TorchCodec even for WAV file inputs.
# This pipeline uses FFmpeg; Colab's preinstalled TorchCodec may not match torch.
subprocess.run([PYTHON, '-m', 'pip', 'uninstall', '-y', 'torchcodec'], check=True)
print('Dependencies installed.')

In [ ]:
# 4. GPUを確認。失敗した場合はGPUランタイムへ変更
GPU_CHECK = r'''
import torch
import torchvision
import torchaudio
from transformers import AutoProcessor
from qwen_tts import Qwen3TTSModel
boxes = torch.tensor([[0., 0., 1., 1.]])
assert torchvision.ops.nms(boxes, torch.tensor([1.]), 0.5).tolist() == [0]
print({'torch': torch.__version__, 'torchvision': torchvision.__version__, 'torchaudio': torchaudio.__version__, 'qwen_import': 'ok'}, flush=True)
assert torch.cuda.is_available(), 'GPUが見つかりません。GPUランタイムへ変更してください。'
props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 1024**3
dtype = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
print({'gpu': props.name, 'vram_gib': round(vram_gib, 2), 'tts_dtype': dtype})
assert vram_gib >= 14, 'VRAMが不足しています。T4（約14.7 GiB）以上が必要です。'
'''
result = subprocess.run([PYTHON, '-u', '-c', GPU_CHECK], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout, flush=True)
result.check_returncode()

## 5. バッチ件数を確認する

A100 80GB向けの初期値はTTS 72件・Whisper 144件です。小さいGPUでは各1件から始めます。


In [ ]:
# 5. バッチ件数。変更する場合はこの2行を編集
TTS_BATCH_SIZE = 72
ASR_BATCH_SIZE = 144
import json
gpu = json.loads(subprocess.check_output(
    [PYTHON, '-c', "import json, torch; p = torch.cuda.get_device_properties(0); print(json.dumps({'name': p.name, 'vram_gib': p.total_memory / 1024**3}))"],
    text=True,
))
assert type(TTS_BATCH_SIZE) is int and TTS_BATCH_SIZE > 0
assert type(ASR_BATCH_SIZE) is int and ASR_BATCH_SIZE > 0
# 再接続で小さいGPUへ変わった場合は、まず各1件に制限
if gpu['vram_gib'] < 70:
    TTS_BATCH_SIZE = ASR_BATCH_SIZE = 1
    print('80GB級GPUではないため、バッチ数を各1件に設定しました。')
os.environ['S2ST_TTS_BATCH_SIZE'] = str(TTS_BATCH_SIZE)
os.environ['S2ST_ASR_BATCH_SIZE'] = str(ASR_BATCH_SIZE)
print({**gpu, 'tts_batch_size': TTS_BATCH_SIZE, 'asr_batch_size': ASR_BATCH_SIZE})

## 6. 5ペアで動作を確認する

日英10音声を生成し、Whisperの品質検査まで通します。採用数と音声を確認してから本番へ進みます。


In [ ]:
CONFIG = REPO_DIR / 'configs' / 'colab-pro.yaml'
# 子プロセスの標準出力・エラーをセルへ転送し、ローカルにも保存
from datetime import datetime
LOG = Path('/content') / ('s2st-smoke-' + datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '.log')
print('Python:', PYTHON, flush=True)
print('Revision:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip(), flush=True)
print('Cache:', os.environ.get('HF_HOME'), 'Models:', os.environ.get('S2ST_MODEL_ROOT'), flush=True)
print('Log:', LOG, flush=True)
with LOG.open('w', encoding='utf-8') as log:
    with subprocess.Popen(
        [PYTHON, '-u', '-m', 's2st_corpus.cli', '--config', str(CONFIG), 'smoke-test'],
        cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
print('returncode:', returncode, flush=True)
if returncode:
    raise RuntimeError(f'Smoke test failed. See the traceback above or {LOG}')

## 7. 保存済みの進捗を確認する

完了済みshard数と次の処理対象を表示します。


In [ ]:
# 現在の進捗
result = subprocess.run(
    [PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG), 'status'],
    cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(result.stdout, flush=True)
result.check_returncode()

## 8. 本番を実行・再開する

`RUN_ALL_REMAINING = False` は次の1 shard、`True` は残りすべてを処理します。全件実行は完了後に集計も行います。中断した場合は同じ設定と保存先で続けます。


In [ ]:
# True: 残り全部 / False: 次の1 shardのみ
RUN_ALL_REMAINING = True
COMMAND = 'run-remaining-shards' if RUN_ALL_REMAINING else 'run-next-shard'
from datetime import datetime
LOG_DIR = DATA_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / ('production-' + datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '.log')
print('Log:', LOG, flush=True)
with LOG.open('w', encoding='utf-8') as log:
    with subprocess.Popen(
        [PYTHON, '-u', '-m', 's2st_corpus.cli', '--config', str(CONFIG), COMMAND],
        cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
print('returncode:', returncode, flush=True)
if returncode:
    raise RuntimeError(f'Production failed. See {LOG}')

## 9. 途中経過を集計する（任意）

完了済みshardから暫定結果を作ります。本番処理を止めた状態で実行してください。


In [ ]:
# 任意：現在までの途中集計
subprocess.run(
    [
        PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG),
        'consolidate', '--allow-incomplete',
    ],
    check=True,
)

## 10. 最終結果を確認する

全shardが終わり、最終集計が必要なときに実行します。`manifests/releases/summary.json` の `complete: true` を確認します。


In [ ]:
# 全shard完了後のみ：正式な最終集計
subprocess.run(
    [PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG), 'consolidate'],
    check=True,
)